# Урок 15. SQL: выборка данных

11 класс · II четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 14](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-14.ipynb) · [Урок 16 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-16.ipynb)

---

SELECT, FROM, WHERE. Условия и логические операторы. DISTINCT. Сортировка ORDER BY и ограничение LIMIT. Практика в sqlite прямо в ноутбуке.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-15", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Язык, на котором спрашивают у данных

С базой не работают, перебирая записи в цикле, — ей задают вопрос
на языке **SQL** (Structured Query Language). Вы описываете, **что**
хотите получить, а как это найти побыстрее, решает сама база.

```sql
SELECT фамилия, класс
FROM ученики
WHERE класс = '11А'
ORDER BY фамилия
```

Читается почти как английская фраза: «выбрать фамилию и класс
из учеников, где класс 11А, упорядочив по фамилии».

SQL придумали в IBM в 1970-х, и с тех пор он понимается почти всеми
базами: SQLite, PostgreSQL, MySQL, Oracle. Различия есть, но основа
везде одна.

### Строение запроса

| Часть | Зачем | Обязательна |
|---|---|---|
| `SELECT` | какие поля вернуть | да |
| `FROM` | из какой таблицы | да |
| `WHERE` | условие отбора строк | нет |
| `ORDER BY` | сортировка | нет |
| `LIMIT` | сколько строк вернуть | нет |

Порядок частей менять нельзя: сначала SELECT, потом FROM, потом
WHERE, потом ORDER BY, в конце LIMIT.

### Условия в WHERE

| Оператор | Пример | Значение |
|---|---|---|
| `=`, `<>` | `класс = '11А'` | равно, не равно |
| `<`, `>`, `<=`, `>=` | `год > 1950` | сравнение |
| `AND`, `OR`, `NOT` | `год > 1950 AND автор = 'Булгаков'` | логика |
| `BETWEEN` | `год BETWEEN 1900 AND 1999` | диапазон, границы включены |
| `IN` | `класс IN ('10А', '11А')` | одно из перечисленных |
| `LIKE` | `название LIKE 'Мастер%'` | шаблон: `%` — любой текст, `_` — один символ |
| `IS NULL` | `класс IS NULL` | значение отсутствует |

Обратите внимание на две вещи. Текст в SQL — в **одинарных** кавычках.
А «не равно» пишется `<>`, хотя большинство баз понимает и `!=`.

### NULL — это не ноль

`NULL` означает «значение неизвестно». Любое сравнение с ним даёт
неизвестный результат, поэтому `WHERE класс = NULL` не найдёт ничего
и никогда. Проверять надо `IS NULL` или `IS NOT NULL`.

### Ещё три полезные вещи

* `SELECT *` — все поля;
* `DISTINCT` — только различные значения:
  `SELECT DISTINCT автор FROM книги`;
* `ORDER BY год DESC` — по убыванию (`ASC` — по возрастанию,
  это и есть значение по умолчанию).

Сортировать можно по нескольким полям сразу:
`ORDER BY класс, фамилия` — сначала по классу, внутри класса
по фамилии.

### Как это выглядит в Python

```python
import sqlite3
соединение = sqlite3.connect("библиотека.db")
курсор = соединение.cursor()
курсор.execute("SELECT * FROM книги WHERE год > 1900")
for строка in курсор.fetchall():
    print(строка)
```

* `execute` отправляет запрос;
* `fetchall()` возвращает список кортежей — по кортежу на строку;
* `fetchone()` возвращает одну строку.

Менять данные (`INSERT`, `UPDATE`, `DELETE`) без `commit()`
бесполезно: изменения не сохранятся.

## Смотрим, как это работает

### Готовим учебную базу

Создадим базу школьной библиотеки — с ней будем работать весь урок.

In [ ]:
import sqlite3

соединение = sqlite3.connect(":memory:")
курсор = соединение.cursor()

курсор.execute("""CREATE TABLE книги (
    id INTEGER PRIMARY KEY, название TEXT, автор TEXT,
    год INTEGER, страниц INTEGER, жанр TEXT)""")

курсор.executemany("INSERT INTO книги VALUES (?, ?, ?, ?, ?, ?)", [
    (1, "Мастер и Маргарита", "Булгаков", 1967, 480, "роман"),
    (2, "Собачье сердце", "Булгаков", 1987, 128, "повесть"),
    (3, "Преступление и наказание", "Достоевский", 1866, 672, "роман"),
    (4, "Идиот", "Достоевский", 1869, 640, "роман"),
    (5, "Му-му", "Тургенев", 1854, 32, "рассказ"),
    (6, "Отцы и дети", "Тургенев", 1862, 288, "роман"),
    (7, "Вишнёвый сад", "Чехов", 1904, 96, "пьеса"),
    (8, "Каштанка", "Чехов", 1887, 48, "рассказ"),
    (9, "Мы", "Замятин", 1924, 224, "роман"),
])
соединение.commit()


def показать(запрос):
    """Выполнить запрос и напечатать результат таблицей."""
    строки = курсор.execute(запрос).fetchall()
    заголовки = [описание[0] for описание in курсор.description]
    print(" | ".join(заголовки))
    print("-" * 60)
    for строка in строки:
        print(" | ".join(str(значение) for значение in строка))
    print(f"[строк: {len(строки)}]\n")


показать("SELECT * FROM книги LIMIT 3")

### Пример 1. Выбор полей и условие

In [ ]:
показать("SELECT название, год FROM книги WHERE год > 1900")

### Пример 2. Составное условие

In [ ]:
показать("""
SELECT название, автор, страниц
FROM книги
WHERE жанр = 'роман' AND страниц < 500
""")

### Пример 3. BETWEEN, IN и LIKE

In [ ]:
показать("SELECT название, год FROM книги WHERE год BETWEEN 1860 AND 1890")
показать("SELECT название, жанр FROM книги WHERE жанр IN ('рассказ', 'пьеса')")
показать("SELECT название FROM книги WHERE название LIKE 'М%'")

`LIKE 'М%'` — начинается на «М». `LIKE '%сад'` — заканчивается
на «сад». `LIKE '%и%'` — содержит «и» где угодно.

### Пример 4. Сортировка и ограничение

In [ ]:
показать("SELECT название, страниц FROM книги ORDER BY страниц DESC LIMIT 3")
показать("SELECT автор, название FROM книги ORDER BY автор, год")

Первый запрос отвечает на вопрос «какие три книги самые толстые»,
второй выстраивает каталог по авторам, а внутри автора — по годам.

### Пример 5. DISTINCT

In [ ]:
показать("SELECT DISTINCT автор FROM книги ORDER BY автор")
показать("SELECT DISTINCT жанр FROM книги")

Без `DISTINCT` в первом запросе Булгаков встретился бы дважды,
Достоевский и Чехов — тоже.

### Пример 6. Одно значение из запроса

Когда нужен не список, а одно число или одна строка, удобен
`fetchone()`.

In [ ]:
самая_старая = курсор.execute(
    "SELECT название, год FROM книги ORDER BY год LIMIT 1").fetchone()
print("Самая старая книга:", самая_старая)

всего = курсор.execute("SELECT COUNT(*) FROM книги").fetchone()[0]
print("Всего книг:", всего)

`COUNT(*)` — первая из агрегатных функций, их подробно разберём
на следующем уроке. `fetchone()` возвращает кортеж, поэтому
и понадобился `[0]`.

## Пробуем сами

Все задачи — к таблице `книги`. В каждой напишите функцию, которая
выполняет запрос и возвращает результат `fetchall()`. Пользуйтесь
готовым `курсор` из ячейки выше.

### Задача 1. Книги одного автора

Верните названия книг Чехова, отсортированные по году (по возрастанию).
Результат — список кортежей из одного элемента.

In [ ]:
def книги_чехова():
    return ...

In [ ]:
si.check("1", книги_чехова, [
    ((), [("Каштанка",), ("Вишнёвый сад",)]),
])

### Задача 2. Книги XIX века

Верните названия и годы книг, изданных с 1800 по 1899 год включительно,
по возрастанию года.

In [ ]:
def девятнадцатый_век():
    return ...

In [ ]:
si.check("2", девятнадцатый_век, [
    ((), [("Му-му", 1854), ("Отцы и дети", 1862),
          ("Преступление и наказание", 1866), ("Идиот", 1869),
          ("Каштанка", 1887)]),
])

### Задача 3. Толстые романы

Верните названия романов объёмом больше 500 страниц,
по убыванию количества страниц.

In [ ]:
def толстые_романы():
    return ...

In [ ]:
si.check("3", толстые_романы, [
    ((), [("Преступление и наказание",), ("Идиот",)]),
])

### Задача 4. Список жанров

Верните различные жанры по алфавиту.

In [ ]:
def жанры():
    return ...

In [ ]:
si.check("4", жанры, [
    ((), [("повесть",), ("пьеса",), ("рассказ",), ("роман",)]),
])

### Задача 5. Поиск по началу названия

Верните названия книг, начинающихся на букву «М», по алфавиту.

In [ ]:
def на_букву_м():
    return ...

In [ ]:
si.check("5", на_букву_м, [
    ((), [("Мастер и Маргарита",), ("Му-му",), ("Мы",)]),
])

### Задача 6. Две самые тонкие книги

Верните название и количество страниц двух самых тонких книг.

In [ ]:
def самые_тонкие():
    return ...

In [ ]:
si.check("6", самые_тонкие, [
    ((), [("Му-му", 32), ("Каштанка", 48)]),
])

### Задача 7. Не Булгаков

Верните названия книг, автор которых **не** Булгаков, по алфавиту.

In [ ]:
def без_булгакова():
    return ...

In [ ]:
si.check("7", без_булгакова, [
    ((), [("Вишнёвый сад",), ("Идиот",), ("Каштанка",), ("Му-му",),
          ("Мы",), ("Отцы и дети",), ("Преступление и наказание",)]),
])

### Задача 8. Что означает NULL

Какое условие найдёт записи, где жанр не указан?

In [ ]:
#@title 🧩 Задача 8. Пустое значение { display-mode: "form" }
#@markdown Выберите ответ
условие = "выбери ответ" #@param ["выбери ответ", "жанр = NULL", "жанр IS NULL", "жанр = ''"]

si.ответ("8", условие, "f23b726bcf6fd4df",
         hint="Сравнение с NULL никогда не бывает истинным.")

## Домашнее задание

### Домашнее задание 1. Романы XX века

Верните название, автора и год романов, изданных после 1900 года,
по возрастанию года.

In [ ]:
def романы_двадцатого():
    return ...

In [ ]:
si.check("дз1", романы_двадцатого, [
    ((), [("Мы", "Замятин", 1924), ("Мастер и Маргарита", "Булгаков", 1967)]),
])

### Домашнее задание 2. Короткое чтение

Верните названия книг короче 150 страниц, у которых жанр — рассказ,
повесть или пьеса, по возрастанию объёма.

In [ ]:
def на_один_вечер():
    return ...

In [ ]:
si.check("дз2", на_один_вечер, [
    ((), [("Му-му",), ("Каштанка",), ("Вишнёвый сад",), ("Собачье сердце",)]),
])

### Домашнее задание 3. Десять вопросов к своей базе

Возьмите схему, которую спроектировали на прошлом уроке, создайте
базу в ноутбуке, наполните её десятком записей и напишите десять
запросов: с условием, с диапазоном, с `LIKE`, с сортировкой,
с `LIMIT` и с `DISTINCT`. К каждому запросу подпишите вопрос
на русском языке, на который он отвечает.

---

### Любопытно

SQL называют декларативным языком: вы говорите, что нужно получить,
а не как это сделать. Внутри базы работает оптимизатор — он смотрит
на размеры таблиц и имеющиеся индексы и сам выбирает план выполнения.
Один и тот же запрос на базе из ста строк и из ста миллионов будет
выполнен по-разному, а вы не меняете в нём ни символа.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 14](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-14.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 16 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-16.ipynb)